In [6]:
import sympy as sp

In [7]:
def calcular_pi_sistema(P):
    n = P.shape[0] # Obtiene la dimensión de la matriz (ej. 4 para 4 clases sociales)
    
    # Crea símbolos pi0, pi1, ..., pin para que SymPy pueda resolver la incógnita
    pi_symbols = sp.symbols(f'pi0:{n}')
    pi = sp.Matrix([pi_symbols])
    
    # Ecuación fundamental: πP - π = 0 
    # Esto busca el vector que no cambia al multiplicarse por la matriz de transición
    eqs = list((pi * P - pi)[0, :])
    
    # Añadir condición de normalización: La suma de todas las probabilidades debe ser 1
    # Sin esta línea, el sistema tendría infinitas soluciones
    eqs.append(sum(pi_symbols) - 1)
    
    # Resuelve el sistema lineal de ecuaciones resultante
    sol = sp.solve(eqs, pi_symbols)
    
    return sol

def calcular_pi_inversa(P):
    """
    Calcula el estado estacionario usando la fórmula: π = jT * (I - P + A)^-1
    Donde A es una matriz de unos e I la identidad.
    """
    n = P.shape[0]
    
    # Prepara las matrices de soporte
    I = sp.eye(n)       # Matriz identidad
    A = sp.ones(n)      # Matriz llena de 1s
    jT = sp.Matrix([[1]*n]) # Vector fila de unos [1, 1, 1, 1]
    
    # Aplica la fórmula de inversión para hallar el vector estacionario
    M = I - P + A
    M_inv = M.inv()
    
    pi = jT * M_inv
    
    # Normaliza el resultado para asegurar que la suma es exactamente 1.0 (100%)
    pi = pi / sum(pi)
    
    return sp.simplify(pi)
def resultado(P):
    try:
        # Intenta obtener el resultado por ambos métodos para validar consistencia
        alpha = calcular_pi_inversa(P)
        beta = calcular_pi_sistema(P)
        return f"Resultado Inversa: {alpha} \nResultado Sistema: {beta}"
    except:
        # Si la matriz inversa falla (ej. matriz singular), intenta solo por sistema
        try:
            alpha = calcular_pi_inversa(P)
            return f"{alpha}"
        except:
            beta = calcular_pi_sistema(P)
            return f"{beta}"

def prob_etapa(matriz_entrada, P, n_iteraciones):
    """
    Calcula la distribución de la población tras 'n' periodos.
    matriz_entrada: El estado inicial (ej. todos vulnerables [1,0,0,0])
    P**n_iteraciones: Eleva la matriz de transición a la potencia del tiempo
    """
    return matriz_entrada * (P**n_iteraciones)

In [8]:
EFRD = sp.Matrix([
    [0.70, 0.25, 0.04, 0.01], # Vulnerable
    [0.15, 0.70, 0.12, 0.03], # Equilibrio
    [0.05, 0.15, 0.75, 0.05], # Consolidado
    [0.01, 0.04, 0.15, 0.80]  # Alto Impacto
])

In [9]:
print("Estado PI:", resultado(EFRD), "\n")

jt = sp.Matrix([[1, 0, 0, 0]])  # Ejemplo en un colapso total. 
                                # El 100% de la población vive con un ingreso menor a k_base.
t = 3 #Los resultados se imprimen tras tres años.
print(f"probabilidad en etapa {t}: {prob_etapa(jt, EFRD, t)}")

Estado PI: Resultado Inversa: Matrix([[0.227592141115732, 0.350781586117883, 0.286103542234333, 0.135522730532052]]) 
Resultado Sistema: {pi0: 0.227592141115732, pi1: 0.350781586117883, pi2: 0.286103542234332, pi3: 0.135522730532052} 

probabilidad en etapa 3: Matrix([[0.428900000000000, 0.396285000000000, 0.134752000000000, 0.0400630000000000]])
